# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [1]:
Threads.nthreads()

32

In [1]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_with_welch_2009.fa"    # "lacS_background.fa", "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa", "Ecoli_example_sequences.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 2
heterologous_genes  = ["hpaBC"] #,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(heterologous_genes, background_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))

## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)

## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs, flags)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs, scale, µ);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);

# ===== DIAGNOSTIC: Check preintegration results =====
println("\n" * "="^80)
println("🔍 PREINTEGRATION DIAGNOSTICS")
println("="^80)
println("Vector length: $(length(y_steady_state))")
println("Minimum value: $(minimum(y_steady_state))")
println("Maximum value: $(maximum(y_steady_state))")
println("Non-finite entries: $(count(!isfinite, y_steady_state))")
println("Values < -1e-9: $(count(x -> x < -1e-9, y_steady_state))")
println("Values < 0: $(count(x -> x < 0, y_steady_state))")

# Find which species are negative
negative_indices = findall(x -> x < -1e-9, y_steady_state)
if !isempty(negative_indices)
    println("\n⚠️  NEGATIVE SPECIES DETECTED:")
    for idx in negative_indices
        if idx <= length(tags_pre.species_and_sequences.Tag)
            tag = tags_pre.species_and_sequences.Tag[idx]
            value = y_steady_state[idx]
            println("  idx=$(idx) tag=$(tag) value=$(value)")
        else
            println("  idx=$(idx) (codon position) value=$(y_steady_state[idx])")
        end
    end
else
    println("\n✅ No negative species detected (threshold: -1e-9)")
end
println("="^80 * "\n")

# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)


# ===== DIAGNOSTIC: Check combined states vector =====
println("\n" * "="^80)
println("🔍 COMBINED STATES VECTOR DIAGNOSTICS")
println("="^80)
println("Vector length: $(length(y0_all))")
println("Minimum value: $(minimum(y0_all))")
println("Maximum value: $(maximum(y0_all))")
println("Non-finite entries: $(count(!isfinite, y0_all))")
println("Values < -1e-9: $(count(x -> x < -1e-9, y0_all))")
println("Values < 0: $(count(x -> x < 0, y0_all))")

# Find which species are negative
negative_indices_all = findall(x -> x < -1e-9, y0_all)
if !isempty(negative_indices_all)
    println("\n⚠️  NEGATIVE SPECIES IN y0_all:")
    # Structure: [shared metabolites] [seq1: S70, rna, PRT, PTC, codons] [seq2: ...] ...
    n_metab_shared = length(tags_pre.species_and_sequences.Tag) - 4  # Shared metabolite pool (excluding last 4)
    
    # Calculate per-sequence structure
    seq_lengths = [length(seqs[seq_name].Codon) - 1 for seq_name in seq_names]
    n_per_seq = [4 + seq_lengths[i] for i in 1:length(seq_names)]  # S70, rna, PRT, PTC + codons per sequence
    
    for idx in negative_indices_all[1:min(20, length(negative_indices_all))]  # Show first 20
        if idx <= n_metab_shared
            tag = tags_pre.species_and_sequences.Tag[idx]
            println("  idx=$(idx) tag=$(tag) value=$(y0_all[idx]) [shared metabolite pool]")
        else
            # Find which sequence this belongs to
            pos = idx - n_metab_shared
            seq_num = 1
            cumsum = 0
            for i in 1:length(n_per_seq)
                if pos <= cumsum + n_per_seq[i]
                    seq_num = i
                    pos_in_seq = pos - cumsum
                    break
                end
                cumsum += n_per_seq[i]
            end
            
            if pos_in_seq <= 4
                tags_seq = ["S70", "rna", "PRT", "PTC"]
                println("  idx=$(idx) seq=$(seq_num) ($(seq_names[seq_num])) $(tags_seq[pos_in_seq]) value=$(y0_all[idx])")
            else
                codon_pos = pos_in_seq - 4
                println("  idx=$(idx) seq=$(seq_num) ($(seq_names[seq_num])) codon_pos=$(codon_pos) value=$(y0_all[idx])")
            end
        end
    end
    if length(negative_indices_all) > 20
        println("  ... and $(length(negative_indices_all) - 20) more negative values")
    end
else
    println("\n✅ No negative species detected in y0_all (threshold: -1e-9)")
end
println("="^80 * "\n")


# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);


# Integration
# PERFORMANCE FIX: Use Diagonal mass matrix instead of dense matrix
# For 100 sequences with ~22k states, dense matrix would be ~4GB and cause O(n³) scaling
n_all = length(y0_all)
mass_diag_all = ones(Float64, n_all)
mass_diag_all[conserved] .= 0.0
M = Diagonal(mass_diag_all)  # Much more memory-efficient: O(n) instead of O(n²)

tspan 	= (0.0, 15.0) ## Define time span for the simulation

p_parallel = local_tags_list

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p_parallel); ## Create an ODE problem



# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (threaded, $(length(seq_names)) sequences)...")

# Tripwire callback: terminate immediately if any state becomes NaN/Inf or very negative
# This prevents "mysterious solver abort" runs and provides clear diagnostics
function condition_tripwire(u, t, integrator)
    # Check for NaN/Inf or very negative values
    neg_tol = -1e-6
    for i in eachindex(u)
        if !isfinite(u[i]) || u[i] < neg_tol
            return true  # Trigger callback
        end
    end
    return false  # Continue integration
end

function affect_tripwire!(integrator)
    u = integrator.u
    t = integrator.t
    neg_tol = -1e-6
    
    # Find the problematic value
    for i in eachindex(u)
        if !isfinite(u[i])
            println("\n⚠️  TRIPWIRE TRIGGERED at t=$(t):")
            println("  Non-finite value detected at index $(i)")
            if i <= length(tags_pre.species_and_sequences.Tag)
                println("  Tag: $(tags_pre.species_and_sequences.Tag[i])")
            end
            println("  Value: $(u[i])")
            terminate!(integrator)
            return
        elseif u[i] < neg_tol
            println("\n⚠️  TRIPWIRE TRIGGERED at t=$(t):")
            println("  Negative value detected at index $(i)")
            if i <= length(tags_pre.species_and_sequences.Tag)
                println("  Tag: $(tags_pre.species_and_sequences.Tag[i])")
            end
            println("  Value: $(u[i]) (threshold: $(neg_tol))")
            terminate!(integrator)
            return
        end
    end
end

cb = DiscreteCallback(condition_tripwire, affect_tripwire!)

# PERFORMANCE FIX: Optimize save settings for large state vectors
# With ~22k states, saving every step is extremely expensive
# Only save start and end unless dense output is needed
sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6, callback=cb,
            save_everystep=false, save_start=false, save_end=true) ## Solve ODE system

# Check solver return code
println("\n" * "="^80)
println("🔍 SOLVER STATUS")
println("="^80)
println("Return code: $(sol.retcode)")
if sol.retcode != :Success
    println("⚠️  WARNING: Solver did not complete successfully!")
    println("  This may indicate numerical issues or constraint violations.")
else
    println("✅ Solver completed successfully")
end
println("="^80 * "\n")

println("Integration completed!")
save_results(sol, local_tags_list, save_data)

🔧 Julia started with 32 threads
Number of sequences: 3

🔍 PREINTEGRATION DIAGNOSTICS
Vector length: 835
Minimum value: 0.0
Maximum value: 129.52079008245448
Non-finite entries: 0
Values < -1e-9: 0
Values < 0: 0

✅ No negative species detected (threshold: -1e-9)


🔍 COMBINED STATES VECTOR DIAGNOSTICS
Vector length: 1100
Minimum value: 0.0
Maximum value: 43.241797675578404
Non-finite entries: 0
Values < -1e-9: 0
Values < 0: 0

✅ No negative species detected in y0_all (threshold: -1e-9)


⏳ Integrating ODE system (threaded, 3 sequences)...

🔍 SOLVER STATUS
Return code: Success
✅ Solver completed successfully

Integration completed!


In [2]:
u = sol.u[end]
du = similar(y_steady_state)
t = sol.t[end]
p = p_parallel 
println(u)


[19.764796851549654, 20.00105796436968, 0.0122304515586147, 0.02177700494776232, 2.3920445342499312, 1.5233429104172804, 0.6827920595049934, 0.7650743757969999, 0.2251031693999853, 2.5482396876565954, 0.09468476138288182, 24.318810957405525, 15.7925588491375, 23.680287230536138, 33.147052428771076, 1.1934896602431057e-6, 0.3980474991026619, 0.24844193477442975, 3.398275494026294, 0.02995777194965004, 0.001118577387107298, 0.0020738956088547952, 0.005489064878247245, 0.0006270756800719765, 0.011435567323397745, 5.5774690974428055e-6, 0.0038103035529895904, 0.00015999845366420918, 0.009747139433258766, 0.017097639198582357, 0.01283903474915617, 0.0007415548634379098, 0.0020962113150180314, 0.000470507081626958, 0.00025655951969262407, 0.02256139913377953, 0.0006494726468306422, 0.0021654644788288366, 0.0013829921309544797, 0.003931813542390076, 0.0002883695556671847, 0.0010705738258981175, 0.048520312153665174, 0.0025699146719633374, 3.562263255576693e-28, 2.228775299255619e-28, 0.000992

In [ ]:
@benchmark solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6)

In [ ]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
t = tspan;

@benchmark ODE_parallel_distributed!(du, u, p, t)


In [ ]:

# Vectors and numbers 
@unpack y_codons_lengths, offsets, n_metab, n_seq, du_metab_all, du_codons, y_codons, results, du_seq, du_metab = p.vectors

# Generate parts of u that need to be assembled specific to the sequence   
@views u_metab =  u[1:n_metab]				                            # Save all pool species from u in a vector 

i = 1

local_tags = p.local_tags_list[i]		                            # Extraction of local tags
idx = n_metab + offsets[i]                                          # Create index to access u on the sequence specific indexes
len = y_codons_lengths[i]            

# thread-safe local views
u_codons = @view u[idx+1 : idx+len]

# assemble without allocation
u_seq = similar(u, n_metab + len)
@views u_seq[1:n_metab] .= u_metab
@views u_seq[n_metab+1:end] .= u_codons

du_seq_local = du_seq[i]					                        # Initialization of sequence specific rates vector

du = du_seq_local
fill!(du, 0.0)
u = u_seq
p = local_tags

# @benchmark rhs_metabolites!(du, u, p)
rhs_metabolites!(du, u, p)


In [ ]:
@benchmark rhs_sequence!(du, u, p)

In [ ]:
function rhs_sequence_test!(du, u, p)

    # du = zeros(Float64, length(u))
    tags = p
    # Unpack parameters 
    @unpack µ, k_proc, k_cat_Pol, C_RNP, V_TCmax, K_M_D, k_cat_E, C_RNE, V_degmax, K_M_deg, k_D, n_DNA_t, n_DNA_s = tags.p_seq.TX
    @unpack k_TLEmaxj, KM_T3j, KM_EFGGTP = tags.p_seq.EL
    @unpack k_TE, KM_RK, KM_GTP, KM_RF3GTP, k_RF3_diss_GDP, k_RF1_diss_GTP, k_GTP_hydr, k_RF1_ass = tags.p_seq.TER
    @unpack k_RF3_GDP_ass, k_RF3_GTP_ass, k_RF3_GDP_diss, k_RF3_GTP_diss = tags.p_seq.RF3
    @unpack A1, A2, A3, A4, A5, k_cat_K_M_G1, k_cat_K_M_G2, k_RRF, q_RRF, one_q_G2_k_split = tags.p_seq.PTC 
    
    # Unpack constant variables 
    ATP, GTP, UTP, CTP, ADP, GDP, UDP, CDP, AMP, GMP, UMP, CMP, Eut, Est, RFt, RRF, Ala, Cys, 
    Asp, Glu, Phe, Gly, His, Ile, Lys, Leu, Met, Asn, Pro, Gln, Arg, Ser, Thr, val, Trp, Tyr, C_D = tags.init_c.constant_vars 

    # Unpack dynamic variables 
    S30, S50, IF1, IF2, IF3, SC1, SC2, SC3, SC4, PIC, EFG, EFD, EGT, EGD, EFT, RF3, RGT, RGD, b1t, ta1, ta2, tc1, td1, te2, tf1, tg2, tg3, th1, ti1, tk1, t1l, t2l, t3l, t4l, t5l, tm1, tn1, tp1, tp2, tp3, 
    tq1, tq2, tr2, tr3, tr4, tr5, ts1, ts2, ts3, ts5, tt1, tt2, tt4, tv1, tv2, tw1, ty1, tA1, tA2, tC1, tD1, tE2, tF1, tG2, tG3, tH1, tI1, tK1, tL1, tL2, tL3, tL4, tL5, tM1, tN1, tP1, tP2, tP3, tQ1, tQ2, 
    tR2, tR3, tR4, tR5, tS1, tS2, tS3, tS5, tT1, tT2, tT4, tV1, tV2, tW1, tY1, a1t, a2t, c1t, d1t, e2t, f1t, g2t, g3t, h1t, i1t, k1t, l1t, l2t, l3t, l4t, l5t, m1t, n1t, p1t, p2t, p3t, q1t, q2t, r2t, r3t, 
    r4t, r5t, s1t, s2t, s3t, s5t, t1t, t2t, t4t, v1t, v2t, w1t, y1t, S70, rna, PRT, PTC = u 


    # Define RIB1
    
    fill!(tags.rate_RIB, 0.0)

    @views c_RIB	= u[tags.idx_first_codon:end]     

    @views total_RNA   = tags.RNA + sum(c_RIB[1:12])

    fill!(tags.q_R_elo, 1.0)
        
    @views for i in 2:(tags.len_sequence - 13)

        if total_RNA-sum(c_RIB[i:i+12]) == 0

            tags.q_R_elo[i] 	= 0;
        else
            tags.q_R_elo[i] 	= 1 - c_RIB[i+12] / (total_RNA-sum(c_RIB[i:i+11]));
        
        end
    end 


    @views for i in (tags.len_sequence-13) : (tags.len_sequence-1)
        if total_RNA - sum(c_RIB[i : end]) == 0
            tags.q_R_elo[i] = 0
        else 
            tags.q_R_elo[i] = 1 - (c_RIB[end] + PTC) / (total_RNA - sum(c_RIB[i:end-1]))
        end
    end

    # dynamic efficiency factor 
    if tags.flags.dynamic_efficiency == true

        efac = dynEffficiencyCalculation(tags, EGT, u);

    end
        

    # V_EL
    C_vect = reshape(u[1:tags.num_species], 1, tags.num_species)  # Directly assign
    c_Codon = tags.t3_Codon_struc * C_vect'

    V_TL_EL	= zeros(tags.len_sequence-1);

    EGT_inv = inv(EGT)


    @inbounds for i in 1:tags.len_sequence-2

        cod_next = tags.species_and_sequences.Seq[i+1]
        cod_curr = tags.species_and_sequences.Seq[i]

        j = tags.species_and_sequences.cod_idx[cod_next]    # O(1) Lookup
        k = tags.species_and_sequences.cod_idx[cod_curr]    # O(1) Lookup

        cc = c_Codon[j]

        if cc == 0
            V_TL_EL[i+1] = 0
        else
            eff = efac[k] * min(tags.efac_pos_aa[i], tags.efac_int_sd[i])

            denom = 1 + KM_T3j/cc + KM_EFGGTP*EGT_inv

            V_TL_EL[i+1] =
                tags.q_R_elo[i] * k_TLEmaxj * c_RIB[i] * eff / denom
        end
    end

    V_TL_EL_sum = sum(V_TL_EL);
    c_codon_seq = c_Codon[tags.matrix_N]

    matrix_free_KT3!(tags, V_TL_EL, c_codon_seq, C_vect)

     
    ## translation termination
    te = tags.len_sequence-1 # Number of last coding codon
    # ke = findfirst(x -> x == tags["Seq1"]["Codon"][te], tags["cod"])

    k_RF1_diss_GTP = k_RF1_diss_GTP#*tags.scale.termination

    efficiency_term = 1;
    k_ter_RF1 = 1 / ( 1/(k_RF1_ass*RFt) + 1/(k_RF1_diss_GTP) );
    k_ter_RF3 = 1 / ( 1/(k_RF1_ass*RGT) + 1/(k_RF3_diss_GDP) + 1/(k_GTP_hydr) );

    if k_ter_RF1 < k_ter_RF3
        k_ter = k_ter_RF1;
    else
        k_ter = k_ter_RF3;
    end


    V_TE_max = k_ter * RFt   # RFt is RF1/2 total
    V_TL_TE = V_TE_max * efficiency_term / (1 + KM_RK/ c_RIB[te] + KM_RF3GTP/RGT + KM_RK * KM_RF3GTP /(c_RIB[te] * RGT))
        
    if c_RIB[te] == 0 || GTP == 0
        V_TL_TE=0;
    end

    V_RF3_GDP_ass  = k_RF3_GDP_ass  * RF3 * GDP;
    V_RF3_GTP_ass  = k_RF3_GTP_ass  * RF3 * GTP;
    V_RF3_GDP_diss = k_RF3_GDP_diss * RGD;
    V_RF3_GTP_diss = k_RF3_GTP_diss * RGT 

    ## post termination complex recycling
    t_rec = A1 + A2 * 1/RRF + A3 * EGT/RRF + A4 * 1/EGT + A5 * 1/(RRF*EGT); # time/s for recycling according to Borg et al. 2016
    k_rec = 1/t_rec; # rate/s^-1 for recycling according to Borg et al. 2016
    k_rec_min = k_rec*60; # rate/min^-1 for recycling according to Borg et al. 2016
    f_s = k_cat_K_M_G1/(k_RRF * RRF)*(q_RRF/k_cat_K_M_G2*one_q_G2_k_split + EGT)+ one_q_G2_k_split; #EGT hydrolised per splitting event according to Borg et al 2016

    # translation termination - calculation of du 

    du[tags.alpha_one[end]] += V_TL_TE

    tags.rate_RIB[tags.len_sequence-1] -= V_TL_TE

    du[tags.idx.PTC] += V_TL_TE - k_rec_min * PTC;	            # PTC
    du[tags.idx.S30] += k_rec_min * PTC;                         # rate of 30S release due to recycling
    du[tags.idx.S50] +=  k_rec_min * PTC;                        # rate of 50S release due to recycling
    du[tags.idx.PRT] = V_TL_TE;                                  # rate of PRT (Protein)
    du[tags.idx.RGT] = -V_TL_TE -V_RF3_GTP_diss +V_RF3_GTP_ass;  # rate of RGT
    du[tags.idx.RGD] =  V_TL_TE -V_RF3_GDP_diss +V_RF3_GDP_ass;  # rate of RGD
    du[tags.idx.RF3] =          +V_RF3_GDP_diss +V_RF3_GTP_diss -V_RF3_GTP_ass -V_RF3_GDP_ass; # rate of RF3


    # translation elongation - calculation of du 

    # concentration of the ribosome at the specific codon
    tags.rate_RIB[1] = - V_TL_EL[2];  

    for i in 2: tags.len_sequence-2
        tags.rate_RIB[i] = V_TL_EL[i] - V_TL_EL[i+1];	
    end

    tags.rate_RIB[tags.len_sequence-1] = tags.rate_RIB[tags.len_sequence-1] + V_TL_EL[end];

    # concentration of the unloaded tRNA
    du[tags.idx.ta1:tags.idx.ty1] .+= tags.K_tRNA[end-tags.species_and_sequences.no_of_tRNAs+1:end]

    # concentration of the ternary-complex
    du[1: tags.num_species] .+= - tags.K_T3;
    du[tags.idx.EGT]  += - V_TL_EL_sum - f_s * k_rec_min * PTC;	# Rate of EFG_GTP 
    du[tags.idx.EGD]  += + V_TL_EL_sum + f_s * k_rec_min * PTC;	# Rate of EFG_GDP

    du[tags.idx.rna] += - V_TL_EL[12];

    # save RIB in du
    du[tags.idx_first_codon:end] .+= tags.rate_RIB;

    return nothing
end

@benchmark rhs_sequence_test!(du, u, p)